<a href="https://colab.research.google.com/github/vadimdddd/CV_task_drawing_2d_find_edges/blob/main/CV_Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Получаем датасет с гугл диска


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

!cp -r /content/gdrive/MyDrive/alpha_version/ .

Установка необходимых модулей для работы


In [ ]:
!pip install --upgrade opencv-python-headless pymupdf numpy matplotlib scikit-image pillow tqdm gradio==6.22.0 --quiet

Импортируем модули

In [ ]:
import os
import cv2
import time
import fitz
import json
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from pathlib import Path

Преобразуем pdf в png для лучшего качества изображений

In [ ]:
def process_pdfs_colab():
    base_dir = Path("/content/gdrive/MyDrive")
    source_dir = base_dir / "alpha_version"
    dest_dir = base_dir / "alpha_version_png"

    print(f"Текущая директория: {Path.cwd()}")
    print(f"Ищем PDF здесь: {source_dir.resolve()}")
    print(f"Сохраним PNG здесь: {dest_dir.resolve()}\n")

    max_wait_time = 30
    wait_interval = 2
    elapsed = 0

    while not source_dir.exists() and elapsed < max_wait_time:
        if elapsed == 0:
            print("Ожидаем появления папки alpha_version...")
        time.sleep(wait_interval)
        elapsed += wait_interval

    if not source_dir.exists():
        print("\nКРИТИЧЕСКАЯ ОШИБКА: Папка /content/gdrive/MyDrive/alpha_version не найдена.")
        return

    dest_dir.mkdir(parents=True, exist_ok=True)

    pdf_files = sorted(source_dir.glob("*.pdf"))

    if not pdf_files:
        print(f"В папке {source_dir} не найдено PDF файлов.")
        return

    saved_images = []

    for idx in tqdm(range(len(pdf_files)), desc="Обработка чертежей", unit="файл", ncols=100):
        pdf_path = pdf_files[idx]
        new_base_name = f"sample_{idx}"

        try:
            doc = fitz.open(str(pdf_path))
            page = doc.load_page(0)

            page_width = page.rect.width
            page_height = page.rect.height

            vertical_rect = fitz.Rect(0, 0, 800, 1000)
            horizontal_rect = fitz.Rect(0, 0, 1800, 1300)

            clip_rect = horizontal_rect if page_width > page_height else vertical_rect

            zoom = 600 / 72
            mat = fitz.Matrix(zoom, zoom)

            pix = page.get_pixmap(matrix=mat, clip=clip_rect, alpha=False)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            png_filename = f"{new_base_name}.png"
            txt_filename = f"{new_base_name}.txt"

            png_path = dest_dir / png_filename
            txt_path = dest_dir / txt_filename

            img.save(png_path, "PNG", optimize=True, compress_level=0)

            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(f"original_name: {pdf_path.name}\n")
                f.write(f"new_name: {png_filename}\n")

            doc.close()
            saved_images.append(img)

        except Exception as e:
            print(f"\nОшибка в файле {pdf_path.name}: {type(e).__name__} - {e}")
            continue

    print(f"\nГотово. Файлы сохранены в: {dest_dir.resolve()}")
    return saved_images

images_list = process_pdfs_colab()

if images_list:
    fig, axes = plt.subplots(nrows=1, ncols=min(5, len(images_list)), figsize=(20, 8))
    if len(images_list) == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i < len(images_list):
            ax.imshow(images_list[i])
            ax.set_title(f'sample_{i}', fontsize=10)
            ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("Список изображений пуст или возникла ошибка при обработке.")

!!!Удалить в будущем, для сохранения памяти просто импортируем уже папку с png

In [ ]:
!cp -r /content/gdrive/MyDrive/alpha_version_png/ .

Выделим регион на изображении, для которого нужно найти контур

In [ ]:
DATA_DIR = Path("/content/gdrive/MyDrive/alpha_version_png")
OUTPUT_JSON = Path("/content/gdrive/MyDrive/roi_data.json")

def auto_detect_roi(image_path):
    img = cv2.imread(str(image_path))
    if img is None: return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    x_bg, y_bg, w_bg, h_bg = cv2.boundingRect(contours[0])

    roi_found = None

    for cnt in contours[1:]:
        area = cv2.contourArea(cnt)
        if area < 5000: continue

        x, y, w, h = cv2.boundingRect(cnt)

        if (x > x_bg and y > y_bg and
            (x+w) < (x_bg+w_bg) and (y+h) < (y_bg+h_bg)):

            roi_found = [x, y, x+w, y+h]
            break

    if not roi_found:
        margin_w, margin_h = int(w_bg * 0.1), int(h_bg * 0.1)
        roi_found = [
            x_bg + margin_w,
            y_bg + margin_h,
            x_bg + w_bg - margin_w,
            y_bg + h_bg - margin_h
        ]

    return roi_found

print("Начинаю автоматический поиск областей интереса...")
roi_dict = {}

if OUTPUT_JSON.exists():
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        try:
            roi_dict = json.load(f)
        except Exception:
            roi_dict = {}

png_files = sorted(DATA_DIR.glob("*.png"))
processed_count = 0

for img_path in tqdm(png_files, desc="Авто-разметка"):
    filename = img_path.name

    if filename in roi_dict:
        processed_count += 1
        continue

    coords = auto_detect_roi(img_path)

    if coords:
        roi_dict[filename] = coords
        processed_count += 1

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(roi_dict, f, indent=4, ensure_ascii=False)

print(f"\nГотово! Размечено {processed_count} новых файлов.")
print(f"Всего в базе: {len(roi_dict)} записей.")

Логика поиска контуров

In [ ]:
def extract_part_contours(image_array, filename):
    try:
        img_bgr = cv2.cvtColor(np.array(image_array), cv2.COLOR_RGB2BGR)
    except Exception:
        img_bgr = np.array(image_array).astype(np.uint8)

    h, w = img_bgr.shape[:2]

    total_area_px = h * w
    MIN_AREA_PERCENT = 0.001
    MAX_AREA_PERCENT = 0.15

    min_area = int(total_area_px * MIN_AREA_PERCENT)
    max_area = int(total_area_px * MAX_AREA_PERCENT)

    margin_x = int(w * 0.05)
    margin_y = int(h * 0.05)

    field_img = img_bgr[margin_y:h-margin_y, margin_x:w-margin_x].copy()
    shift_x, shift_y = margin_x, margin_y

    gray = cv2.cvtColor(field_img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    _, binary = cv2.threshold(enhanced, 220, 255, cv2.THRESH_BINARY_INV)

    kernel_thin = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (4, 4))
    thinned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_thin, iterations=3)

    contours, _ = cv2.findContours(thinned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    output_vis = img_bgr.copy()

    if not contours:
        return Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)), \
               Image.fromarray(cv2.cvtColor(output_vis, cv2.COLOR_BGR2RGB))

    sorted_cnts = sorted(contours, key=cv2.contourArea, reverse=True)

    final_polylines = []

    for cnt in sorted_cnts:
        area = cv2.contourArea(cnt)

        if area < min_area: break
        if area > max_area: continue

        x, y, cw, ch = cv2.boundingRect(cnt)
        rect_area = cw * ch

        fill_ratio = area / rect_area
        if fill_ratio < 0.25:
            continue

        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.01 * peri, True)

        if len(approx) < 6:
            continue

        shifted_approx = approx + np.array([shift_x, shift_y])
        final_polylines.append(shifted_approx)

    LINE_THICKNESS = 10

    if final_polylines:
        cv2.polylines(output_vis, final_polylines, isClosed=True, color=(0, 255, 0), thickness=LINE_THICKNESS)

    original_out = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    result_out = Image.fromarray(cv2.cvtColor(output_vis, cv2.COLOR_BGR2RGB))
    return original_out, result_out

Сохранение результатов обработки

In [ ]:
OUTPUT_DIR = Path("/content/gdrive/MyDrive/results")

def create_comparison_image(original_img, processed_img, filename):
    """
    Создает изображение-пару: оригинал слева, результат справа.
    """
    target_h = 800

    orig_w, orig_h = original_img.size
    proc_w, proc_h = processed_img.size

    scale_orig = target_h / orig_h
    scale_proc = target_h / proc_h

    new_orig_w = int(orig_w * scale_orig)
    new_proc_w = int(proc_w * scale_proc)

    original_resized = original_img.resize((new_orig_w, target_h), Image.Resampling.LANCZOS)
    processed_resized = processed_img.resize((new_proc_w, target_h), Image.Resampling.LANCZOS)

    total_w = new_orig_w + new_proc_w
    comparison_img = Image.new('RGB', (total_w, target_h), color='white')

    comparison_img.paste(original_resized, (0, 0))
    comparison_img.paste(processed_resized, (new_orig_w, 0))

    return comparison_img

if not DATA_DIR.exists():
    print(f"Папка с входными данными не найдена: {DATA_DIR}")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    png_files = sorted(DATA_DIR.glob("*.png"))

    if not png_files:
        print("В папке alpha_version_png не найдено PNG-файлов.")
    else:
        print(f"Найдено файлов: {len(png_files)}")
        success_count = 0

        for img_path in tqdm(png_files, desc="Пакетная обработка"):
            try:
                pil_image = Image.open(img_path).convert("RGB")

                original_out, result_out = extract_part_contours(pil_image, img_path.name)

                final_image = create_comparison_image(original_out, result_out, img_path.name)

                save_path = OUTPUT_DIR / f"result_{img_path.name}"
                final_image.save(save_path, format="PNG", optimize=True)
                success_count += 1

            except Exception as e:
                print(f"\nОшибка обработки {img_path.name}: {e}")
                continue

        print(f"\nГотово! Обработано успешно: {success_count}/{len(png_files)}")
        print(f"Результаты лежат здесь: {OUTPUT_DIR}")

Интерфейс демонстрации

In [ ]:
TEMP_PDF_RENDER = "/tmp/gradio_pdf_render.png"

def render_pdf_to_image(pdf_file_path):
    try:
        doc = fitz.open(str(pdf_file_path))
        if doc.page_count == 0:
            return None
        page = doc.load_page(0)
        zoom_x = zoom_y = 300 / 72
        mat = fitz.Matrix(zoom_x, zoom_y)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        pix.save(TEMP_PDF_RENDER)
        return Image.open(TEMP_PDF_RENDER).convert("RGB")
    except Exception as e:
        print(f"Ошибка чтения PDF: {e}")
        return None

def process_wrapper(file_input):
    if file_input is None:
        return None, None

    img_obj = None
    filename = "uploaded_sketch.png"

    if isinstance(file_input, bytes):
        temp_upload_path = "/tmp/user_uploaded.pdf"
        with open(temp_upload_path, "wb") as f:
            f.write(file_input)

        img_obj = render_pdf_to_image(temp_upload_path)
        filename = "uploaded_pdf_page1.png"

    elif isinstance(file_input, str):
        try:
            img_obj = Image.open(file_input).convert("RGB")

            p = Path(file_input)
            if p.exists():
                filename = p.name
            else:
                filename = "direct_upload.png"
        except Exception as e:
            print(f"Не удалось открыть файл по пути {file_input}: {e}")
            return None, None

    elif hasattr(file_input, 'mode'):
        img_obj = file_input
        if hasattr(file_input, 'name'):
            filename = os.path.basename(file_input.name)

    if img_obj is None:
        return None, None

    return extract_part_contours(img_obj, filename)

with gr.Blocks() as demo:
    gr.Markdown("# Детекция контура детали\nПоддерживаемые форматы: перетащите сюда ваш **.png**, **.jpg** или **.pdf**.")

    with gr.Row():
        input_img = gr.Image(
            type="pil",
            label="Исходный чертеж",
            height=400,
            interactive=True
        )
        result_img = gr.Image(type="pil", label="Результат", height=400)

    def get_examples_list():
        examples = []
        if DATA_DIR.exists():
            png_files = sorted(DATA_DIR.glob("*.png"))[:10]
            examples = [[str(f)] for f in png_files]
        return examples

    examples_list = get_examples_list()
    if examples_list:
        gr.Examples(
            examples=examples_list,
            inputs=[input_img],
            outputs=[result_img],
            fn=lambda x: process_wrapper(x),
            cache_examples=False
        )

    run_btn = gr.Button("Найти контур", variant="primary", size="lg")
    run_btn.click(fn=process_wrapper, inputs=input_img, outputs=[input_img, result_img])

demo.launch(debug=True)